<a href="https://colab.research.google.com/github/deboraqlopes/desafio_busca-semantica-ibre/blob/main/Desafio_t%C3%A9cnico_FGV_IBRE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Desafio Técnico**
## Estágio em Ciência de Dados

## **Etapa 1** - Limpeza e tratamento dos dados

In [ ]:
!pip install sentence-transformers beautifulsoup4 -q

In [ ]:
import requests
import json

url = "https://raw.githubusercontent.com/MateusRestier/desafio-ciencia-de-dados-ibre/main/dados/noticias_brutas.json"
noticias = json.loads(requests.get(url).text)

print(f"Total de notícias: {len(noticias)}")
print("\nExemplo bruto:")
print(noticias[0]['texto'][:300])

In [ ]:
import re
from bs4 import BeautifulSoup

def limpar_texto(texto):
    # Remover tags HTML e entidades
    soup = BeautifulSoup(texto, "html.parser")
    texto = soup.get_text(separator=" ")

    # Remover datas de publicação vazadas no corpo do texto
    texto = re.sub(r'[Pp]ublicado\s+em:?\s*[\d/]+\s*(às|as)?\s*[\dh]+\s*', '', texto)
    texto = re.sub(r'\d{2}/\d{2}/\d{4}\s*[-—]?\s*[\dh:]*\s*\|?\s*[A-Za-zÀ-ÿ\s]*?(?=\s{2,}|\s[A-ZÁÉÍÓÚÀÂÊÔÃÕÇ])', '', texto)

    # Remover outros padrões de metadados comuns ex: "[tags: economia]"
    texto = re.sub(r'\[.*?\]', '', texto)

    # Remover múltiplas quebras de linha
    texto = re.sub(r'\n+', ' ', texto)

    # Remover espaços múltiplos
    texto = re.sub(r' +', ' ', texto)

    # Remover espaços nas bordas
    texto = texto.strip()

    return texto

In [ ]:
# Aplicar a limpeza em todas as notícias
dados_limpos = []
for noticia in noticias:
    dados_limpos.append({
        "id": noticia["id"],
        "titulo": noticia["titulo"],
        "texto": limpar_texto(noticia["texto"]),
        "data": noticia["data"],
        "fonte": noticia["fonte"]
    })

# Comparar antes e depois para verificar
print("=== ANTES ===")
print(noticias[0]['texto'][:300])
print("\n=== DEPOIS ===")
print(dados_limpos[0]['texto'][:300])

In [ ]:
print("Tamanho dos textos limpos:\n")
for n in dados_limpos:
    tamanho = len(n['texto'])
    alerta = " ⚠️ CONTEÚDO MÍNIMO" if tamanho < 100 else ""
    print(f"ID {n['id']:2d} | {tamanho:4d} chars | {n['titulo'][:50]}{alerta}")

In [ ]:
print("=== ANTES (bruto) ===")
print(repr(noticias[17]['texto']))

print("\n=== DEPOIS (limpo) ===")
print(repr(dados_limpos[17]['texto']))

In [ ]:
# Marcando como inválido textos com menos de 100 caracteres
for noticia in dados_limpos:
    noticia['valido'] = len(noticia['texto']) >= 100

# Conferindo
for n in dados_limpos:
    if not n['valido']:
        print(f"ID {n['id']} marcada como inválida: '{n['texto']}'")

In [ ]:
padroes = {
    "timestamp dd/mm/aaaa": r'\d{2}/\d{2}/\d{4}',
    "tag HTML restante":    r'<[^>]+>',
    "entidade HTML":        r'&[a-zA-Z]+;',
    "metadado embutido":    r'\[.*?\]',
    "espaços múltiplos":    r' {2,}',
}

print("Verificando resíduos em todos os textos limpos:\n")
encontrou_algo = False

for noticia in dados_limpos:
    if not noticia['valido']:
        continue
    for nome, padrao in padroes.items():
        matches = re.findall(padrao, noticia['texto'])
        if matches:
            encontrou_algo = True
            print(f"ID {noticia['id']:2d} | {nome}: {matches}")

if not encontrou_algo:
    print("Nenhum resíduo encontrado!")

In [ ]:
# Salvando o arquivo
with open('dados_limpos.json', 'w', encoding='utf-8') as f:
    json.dump(dados_limpos, f, ensure_ascii=False, indent=2)

print("\ndados_limpos.json salvo com sucesso!")
print(f"Total: {len(dados_limpos)} notícias ({sum(1 for n in dados_limpos if n['valido'])} válidas, {sum(1 for n in dados_limpos if not n['valido'])} inválidas)")

##**Etapa 2**- Geração de embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Modelo carregado")

In [ ]:
import numpy as np

# Selecionando as notícias válidas
noticias_validas = [n for n in dados_limpos if n['valido']]

# Combinando título + texto para embeddings mais ricos
corpus = [f"{n['titulo']}. {n['texto']}" for n in noticias_validas]

# Gerando os embeddings
embeddings = modelo.encode(corpus, show_progress_bar=True)

print(f"\nEmbeddings gerados!")
print(f"Shape: {embeddings.shape}  →  {len(noticias_validas)} notícias x {embeddings.shape[1]} dimensões")

In [ ]:
np.save('embeddings.npy', embeddings)

# Salvando também o mapeamento id -> posição no array
mapeamento = [{"posicao": i, "id": n["id"], "titulo": n["titulo"]}
              for i, n in enumerate(noticias_validas)]

with open('mapeamento.json', 'w', encoding='utf-8') as f:
    json.dump(mapeamento, f, ensure_ascii=False, indent=2)

print("embeddings.npy salvo!")
print("mapeamento.json salvo!")

## **Etapa 3**- Motor de Busca Semântico

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar(query, top_k=3):
    # Transformar a query em embedding
    embedding_query = modelo.encode([query])

    # Calcular similaridade com todas as notícias
    similaridades = cosine_similarity(embedding_query, embeddings)[0]

    # Ordenar do mais similar para o menos
    indices_ordenados = np.argsort(similaridades)[::-1][:top_k]

    # Retornar os resultados
    print(f"\nQuery: '{query}'")
    print("=" * 60)
    for i, idx in enumerate(indices_ordenados):
        noticia = noticias_validas[idx]
        score = similaridades[idx]
        print(f"\n#{i+1} | Score: {score:.4f}")
        print(f"Título: {noticia['titulo']}")
        print(f"Texto: {noticia['texto'][:150]}...")

In [ ]:
queries = [
    "mudanças na taxa de juros",
    "mercado de trabalho e desemprego",
    "inflação e preços ao consumidor"
]

for query in queries:
    buscar(query)
    print()